In [ ]:
# Team starter - load the unchanged primary split and the chronological robustness split

from pathlib import Path
import pandas as pd
import joblib

from sklearn.base import clone
from sklearn.pipeline import Pipeline

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

df = pd.read_csv(project_root / "data" / "hotel_bookings_clean.csv")
train_ids = pd.read_csv(project_root / "data" / "splits" / "train_ids.csv")["row_id"]
test_ids = pd.read_csv(project_root / "data" / "splits" / "test_ids.csv")["row_id"]
chronological_split = pd.read_csv(
    project_root / "data" / "splits" / "chronological_split.csv",
    parse_dates=["scheduled_arrival_date"]
)
preprocessor = joblib.load(
    project_root / "results" / "models" / "preprocessor_template.joblib"
)

# Primary model-comparison data: unchanged stratified random 80/20 split.
train_df = df[df["row_id"].isin(train_ids)].copy()
test_df = df[df["row_id"].isin(test_ids)].copy()

model_features = []
for name, transformer, columns in preprocessor.transformers:
    model_features.extend(list(columns))

X_train = train_df[model_features]
X_test = test_df[model_features]
y_train = train_df["is_canceled"]
y_test = test_df["is_canceled"]

# Additional robustness data: scheduled-arrival chronological split only.
chrono_train_ids = chronological_split.loc[chronological_split["split"] == "train", "row_id"]
chrono_test_ids = chronological_split.loc[chronological_split["split"] == "test", "row_id"]
chrono_train_df = df[df["row_id"].isin(chrono_train_ids)].copy()
chrono_test_df = df[df["row_id"].isin(chrono_test_ids)].copy()

X_chrono_train = chrono_train_df[model_features]
X_chrono_test = chrono_test_df[model_features]
y_chrono_train = chrono_train_df["is_canceled"]
y_chrono_test = chrono_test_df["is_canceled"]

assert X_train.shape == (94851, 24) and X_test.shape == (23713, 24)
assert X_chrono_train.shape == (96428, 24) and X_chrono_test.shape == (22136, 24)

print("Primary random split:", X_train.shape, X_test.shape)
print("Chronological robustness split:", X_chrono_train.shape, X_chrono_test.shape)
print("Scheduled-arrival boundary: train through 2017-04-30; test from 2017-05-01")


In [ ]:

# Step 3: Smoke test with Dummy Classifier

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy_pipeline = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", DummyClassifier(strategy="most_frequent"))
])

# Fit using training data only
dummy_pipeline.fit(X_train, y_train)

# Predict on held-out test data
y_pred = dummy_pipeline.predict(X_test)

print("Dummy Classifier smoke test")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("F1 score:", round(f1_score(y_test, y_pred), 4))
print("Predictions:", len(y_pred))
print("Pipeline test passed.")


In [ ]:
# Final handover verification

required_files = [
    "README.md",
    "data/hotel_bookings_clean.csv",
    "data/data_dictionary.csv",
    "data/handover_summary.csv",
    "data/splits/train_ids.csv",
    "data/splits/test_ids.csv",
    "data/splits/chronological_split.csv",
    "results/models/preprocessor_template.joblib",
    "notebooks/A_08_Team_Starter.ipynb"
]

missing = []
for file in required_files:
    path = project_root / file
    status = "OK" if path.is_file() else "MISSING"
    print(f"{status}: {file}")
    if not path.is_file():
        missing.append(file)

assert not missing, f"Missing handover files: {missing}"
print("\nFINAL CHECK PASSED")
print("Shared data foundation is ready.")
